# Parte 1 – Scraping de decretos de emergencia

**Assignment 2 – Scraping, APIs y cruce por ubigeo**
**Grupo 1 · Temporada asignada: 1 de enero al 31 de mayo de 2022**

En este notebook reunimos los decretos supremos de la PCM que declaran (o prorrogan) el Estado de Emergencia por lluvias, usando el buscador de normas de gob.pe. Como la página de búsqueda carga los resultados con JavaScript, usamos **Selenium**; las páginas individuales de cada norma se leen con **requests** y **BeautifulSoup**.

Productos de este notebook:

- `datos/decretos_lluvias.csv`
- `datos/decretos_por_departamento.csv`

In [1]:
import os
import re
import time
from urllib import robotparser

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

# gob.pe rechaza los pedidos que no se identifican como navegador
HEADERS = {"User-Agent": "Mozilla/5.0"}

# URL base de búsqueda (dada en el enunciado)
URL_BASE = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
            "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent")

PAUSA_SELENIUM = 3   # segundos entre páginas de búsqueda (el enunciado pide al menos 2)
PAUSA_REQUESTS = 1   # segundos entre páginas de normas (el enunciado pide 1)

## Paso 1. `robots.txt`

Descargamos e imprimimos el archivo `robots.txt` de gob.pe.

In [2]:
respuesta_robots = requests.get("https://www.gob.pe/robots.txt", headers=HEADERS, timeout=30)
print("Código de estado:", respuesta_robots.status_code)
print()
print(respuesta_robots.text)

Código de estado: 200

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



Comprobamos lo que leímos con `urllib.robotparser`, el lector de `robots.txt` de la librería estándar de Python:

In [3]:
lector_robots = robotparser.RobotFileParser()
lector_robots.parse(respuesta_robots.text.splitlines())

url_ejemplo = URL_BASE + "&desde=01-01-2022&hasta=31-01-2022"

print("¿Cualquier bot (*) puede leer nuestra búsqueda en /busquedas?:", lector_robots.can_fetch("*", url_ejemplo))
print("¿Cualquier bot (*) puede entrar a /admin/?:                  ", lector_robots.can_fetch("*", "https://www.gob.pe/admin/"))
print("¿AhrefsBot puede leer nuestra búsqueda?:                     ", lector_robots.can_fetch("AhrefsBot", url_ejemplo))
print()
print("Crawl-delay pedido a GPTBot:       ", lector_robots.crawl_delay("GPTBot"), "segundos")
print("Crawl-delay pedido a OAI-SearchBot:", lector_robots.crawl_delay("OAI-SearchBot"), "segundos")
print("Crawl-delay pedido a *:            ", lector_robots.crawl_delay("*"))
print()
# robotparser NO entiende el comodín "*" dentro de las rutas (Disallow: /*&sheet=),
# por eso esa regla la revisamos a mano: nuestra URL no debe tener el parámetro "sheet=".
print("¿Nuestra URL usa el parámetro prohibido 'sheet='?:", "sheet=" in url_ejemplo)

¿Cualquier bot (*) puede leer nuestra búsqueda en /busquedas?: True
¿Cualquier bot (*) puede entrar a /admin/?:                   False
¿AhrefsBot puede leer nuestra búsqueda?:                      False

Crawl-delay pedido a GPTBot:        5 segundos
Crawl-delay pedido a OAI-SearchBot: 2 segundos
Crawl-delay pedido a *:             None

¿Nuestra URL usa el parámetro prohibido 'sheet='?: False


**¿Qué prohíbe el `robots.txt`?**

- Para **todos los bots** (`User-agent: *`) prohíbe tres cosas:
  - `Disallow: /admin/` → el panel de administración del sitio.
  - `Disallow: /*?sheet=` y `Disallow: /*&sheet=` → cualquier URL que tenga el parámetro `sheet`. El `*` es un comodín: vale para cualquier ruta. Al explorar la página nos dimos cuenta de que **`sheet` es justamente el parámetro de paginación del buscador**: al hacer clic en "Siguiente" la URL pasa a terminar en `&sheet=2`, `&sheet=3`, etc. Es decir, el sitio **no quiere que los bots recorran las páginas 2, 3, … de los resultados**.
- A `AhrefsBot` (un bot de SEO) le prohíbe **todo el sitio** (`Disallow: /`).
- A `GPTBot` y `OAI-SearchBot` (bots de OpenAI) les aplica las mismas prohibiciones que a todos, y además les pide un `Crawl-delay` de **5** y **2 segundos**, respectivamente, entre pedido y pedido.

**¿Está permitida la página de búsquedas (`/busquedas`)?** Sí. Ninguna regla `Disallow` empieza con `/busquedas`, y nuestras URLs no usan el parámetro `sheet`, así que para un bot genérico (`*`) están permitidas. `robotparser` lo confirma (`True`). Ojo: `robotparser` no entiende el comodín `*` dentro de las rutas, así que la regla de `sheet` la revisamos a mano.

**Consecuencia práctica para nuestro scraping:** como no debemos usar `sheet=`, **no vamos a pasar de página** con el botón "Siguiente". Por eso buscamos **mes por mes**: cada búsqueda mensual trae menos de 25 resultados (el máximo por página), así que todo entra en la primera página. Además, al probar el botón "Siguiente" vimos que la nueva URL **pierde el filtro `institucion[]=pcm`**, así que tampoco sería confiable. La tabla de verificación del paso 6 nos avisará si algún mes tuviera más resultados de los que caben en una página.

**¿Por qué igual conviene hacer pausas entre página y página?** Aunque para `*` no se pide un `Crawl-delay`, el sitio sí se lo pide a otros bots (2 y 5 segundos). Eso nos dice cuál es el ritmo que el sitio considera aceptable. gob.pe es un servicio público: si le mandamos muchos pedidos seguidos consumimos recursos que usan los ciudadanos, y además nos arriesgamos a que el servidor nos bloquee (ya vimos que rechaza los pedidos sin `User-Agent`). Por eso hacemos una pausa de **3 segundos** entre páginas de búsqueda con Selenium y de **1 segundo** entre páginas de normas con `requests`.

## Paso 2. Abrir la búsqueda de un solo mes con Selenium

Primero definimos los cinco meses de la temporada 2022 con sus fechas `desde` y `hasta` (formato `DD-MM-AAAA`, como pide la URL).

In [4]:
meses = [
    {"mes": "2022-01 (enero)",   "desde": "01-01-2022", "hasta": "31-01-2022"},
    {"mes": "2022-02 (febrero)", "desde": "01-02-2022", "hasta": "28-02-2022"},
    {"mes": "2022-03 (marzo)",   "desde": "01-03-2022", "hasta": "31-03-2022"},
    {"mes": "2022-04 (abril)",   "desde": "01-04-2022", "hasta": "30-04-2022"},
    {"mes": "2022-05 (mayo)",    "desde": "01-05-2022", "hasta": "31-05-2022"},
]
pd.DataFrame(meses)

,mes,desde,hasta
0,2022-01 (enero),01-01-2022,31-01-2022
1,2022-02 (febrero),01-02-2022,28-02-2022
2,2022-03 (marzo),01-03-2022,31-03-2022
3,2022-04 (abril),01-04-2022,30-04-2022
4,2022-05 (mayo),01-05-2022,31-05-2022


Abrimos el navegador. Usamos **Microsoft Edge** (es el navegador instalado en la computadora; Selenium descarga solo el driver que corresponde). Lo abrimos en modo *headless* (sin ventana) para que el notebook corra de principio a fin sin interrupciones; si se quiere ver el navegador, basta con borrar la línea `--headless=new`.

La función `abrir_busqueda` arma la URL del mes y usa **`WebDriverWait`** para esperar a que JavaScript pinte el contador de resultados (el título que dice "N Resultados") o, si la búsqueda no encuentra nada, el aviso "No se encontraron resultados" (en ese caso no aparece el contador y el total es 0). Así no dependemos de un `time.sleep()` fijo: el código sigue apenas la página está lista (o da error si en 30 segundos no carga).

In [5]:
opciones = webdriver.EdgeOptions()
opciones.add_argument("--headless=new")          # borrar esta línea para ver el navegador
opciones.add_argument("--window-size=1400,1000")
driver = webdriver.Edge(options=opciones)
espera = WebDriverWait(driver, 30)


def abrir_busqueda(desde, hasta, url_base=URL_BASE):
    # Abre la búsqueda para un rango de fechas y espera a que carguen los resultados
    url = url_base + f"&desde={desde}&hasta={hasta}"
    driver.get(url)
    # Esperamos hasta que aparezca el contador "N Resultados"
    # o, si no hay nada, el aviso "No se encontraron resultados"
    espera.until(EC.any_of(
        EC.text_to_be_present_in_element((By.CSS_SELECTOR, "h2[role='status']"), "Resultado"),
        EC.text_to_be_present_in_element((By.CSS_SELECTOR, "main h2"), "No se encontraron"),
    ))
    # Si hay resultados, esperamos también a que aparezca al menos un <article>
    if leer_total() > 0:
        espera.until(EC.presence_of_element_located((By.TAG_NAME, "article")))
    return url


def leer_total():
    # Lee el número de "N Resultados" que muestra la página
    contador = driver.find_elements(By.CSS_SELECTOR, "h2[role='status'] span")
    if not contador:   # la página dice "No se encontraron resultados"
        return 0
    return int(re.sub(r"\D", "", contador[0].text))


mes_prueba = meses[0]
url_prueba = abrir_busqueda(mes_prueba["desde"], mes_prueba["hasta"])
print("URL abierta:", url_prueba)
print("Título de la pestaña:", driver.title)

URL abierta: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2022&hasta=31-01-2022
Título de la pestaña: Plataforma del Estado Peruano


## Paso 3. Número total de resultados

In [6]:
texto_contador = driver.find_element(By.CSS_SELECTOR, "h2[role='status'] span").text
total_prueba = leer_total()
print(f"Texto que muestra la página: '{texto_contador}'")
print(f"Número total de resultados en {mes_prueba['mes']}: {total_prueba}")

Texto que muestra la página: '8 Resultados'
Número total de resultados en 2022-01 (enero): 8


## Paso 4. Extraer los datos de cada resultado

Cada resultado es un `<article>`. Primero encontramos todos los `article` de la página y luego, **dentro de cada uno**, buscamos:

| Dato | Dónde está dentro del `<article>` |
|---|---|
| `numero` | texto del enlace principal (`<a class="link-principal">`) |
| `fecha` | el `<span>` cuyo texto empieza con `Publicado:` |
| `titulo_corto` | el párrafo `<p>` que está justo debajo de la fecha |
| `enlace` | el atributo `href` del enlace principal |

Para la fecha y el título usamos XPath: `.//span[starts-with(normalize-space(.), 'Publicado:')]` busca el `span` que empieza con "Publicado:" y `following-sibling::p` toma el párrafo que le sigue. El punto inicial (`.//`) es importante: hace que la búsqueda sea **dentro de ese artículo** y no en toda la página.

In [7]:
XPATH_FECHA = ".//span[starts-with(normalize-space(.), 'Publicado:')]"


def extraer_resultados():
    # Recorre todos los <article> de la página y saca los 4 datos de cada uno
    filas = []
    articulos = driver.find_elements(By.TAG_NAME, "article")
    for articulo in articulos:
        enlace_principal = articulo.find_element(By.CSS_SELECTOR, "a.link-principal")
        fecha = articulo.find_element(By.XPATH, XPATH_FECHA)
        titulo = articulo.find_element(By.XPATH, XPATH_FECHA + "/following-sibling::p")
        filas.append({
            "numero": enlace_principal.text.strip(),
            "fecha_texto": fecha.text.strip(),
            "titulo_corto": titulo.text.strip(),
            "enlace": enlace_principal.get_attribute("href"),
        })
    return filas


resultados_prueba = extraer_resultados()
print(f"Resultados extraídos en {mes_prueba['mes']}: {len(resultados_prueba)} (la página dice {total_prueba})")
pd.DataFrame(resultados_prueba)

Resultados extraídos en 2022-01 (enero): 8 (la página dice 8)


,numero,fecha_texto,titulo_corto,enlace
0,Decreto Supremo N.° 011-2022-PCM,Publicado: 30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia ...,https://www.gob.pe/institucion/pcm/normas-legales/2724291-011-2022-pcm
1,Decreto Supremo N.° 010-2022-PCM,Publicado: 29 de enero de 2022,"Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir ...",https://www.gob.pe/institucion/pcm/normas-legales/2723437-010-2022-pcm
2,Decreto Supremo N.° 009-2022-PCM,Publicado: 28 de enero de 2022,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto...",https://www.gob.pe/institucion/pcm/normas-legales/2817913-009-2022-pcm
3,Decreto Supremo N.° 004-2022-PCM,Publicado: 25 de enero de 2022,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria...",https://www.gob.pe/institucion/pcm/normas-legales/2810643-004-2022-pcm
4,Decreto Supremo N.° 007-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,https://www.gob.pe/institucion/pcm/normas-legales/2817884-007-2022-pcm
5,Decreto Supremo N.° 006-2022-PCM,Publicado: 18 de enero de 2022,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de...",https://www.gob.pe/institucion/pcm/normas-legales/2817856-006-2022-pcm
6,Decreto Supremo N.° 005-2022-PCM,Publicado: 16 de enero de 2022,Decreto Supremo que modifica el Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunsta...,https://www.gob.pe/institucion/pcm/normas-legales/2677615-005-2022-pcm
7,Decreto Supremo N.° 002-2022-PCM,Publicado: 6 de enero de 2022,"Decreto Supremo que modifica el numeral 8.1 del artículo 8 del Decreto Supremo N° 184-2020-PCM, Decreto Supremo que declara Estado de Emergencia N...",https://www.gob.pe/institucion/pcm/normas-legales/2622790-002-2022-pcm


## Paso 5. Repetir para todos los meses de la temporada

Repetimos los pasos 2 a 4 para los cinco meses. Entre página y página hacemos una pausa de `PAUSA_SELENIUM` = 3 segundos. Guardamos, para cada mes, el total que dice la página y cuántos resultados extrajimos, para verificarlos en el paso 6.

In [8]:
todas_las_filas = []
verificacion = []

for m in meses:
    abrir_busqueda(m["desde"], m["hasta"])
    total = leer_total()
    filas = extraer_resultados()
    for fila in filas:
        fila["mes"] = m["mes"]
    todas_las_filas.extend(filas)
    verificacion.append({"mes": m["mes"], "resultados_totales": total, "resultados_extraidos": len(filas)})
    print(f"{m['mes']}: la página dice {total} resultados, extrajimos {len(filas)}")
    time.sleep(PAUSA_SELENIUM)   # pausa entre página y página

driver.quit()   # cerramos el navegador

normas = pd.DataFrame(todas_las_filas)
print(f"\nTotal de filas extraídas: {len(normas)}")

2022-01 (enero): la página dice 8 resultados, extrajimos 8
2022-02 (febrero): la página dice 5 resultados, extrajimos 5
2022-03 (marzo): la página dice 5 resultados, extrajimos 5
2022-04 (abril): la página dice 4 resultados, extrajimos 4
2022-05 (mayo): la página dice 7 resultados, extrajimos 7

Total de filas extraídas: 29


## Paso 6. Verificación: resultados totales vs. extraídos

In [9]:
tabla_verificacion = pd.DataFrame(verificacion)
tabla_verificacion["coinciden"] = tabla_verificacion["resultados_totales"] == tabla_verificacion["resultados_extraidos"]
print(tabla_verificacion.to_string(index=False))
print()
print("TOTAL", tabla_verificacion["resultados_totales"].sum(), "vs", tabla_verificacion["resultados_extraidos"].sum())

assert tabla_verificacion["coinciden"].all(), "¡Hay meses donde no coinciden! Revisar el código antes de seguir."
print("\nOK: en todos los meses coinciden el total de la página y lo extraído.")

              mes  resultados_totales  resultados_extraidos  coinciden
  2022-01 (enero)                   8                     8       True
2022-02 (febrero)                   5                     5       True
  2022-03 (marzo)                   5                     5       True
  2022-04 (abril)                   4                     4       True
   2022-05 (mayo)                   7                     7       True

TOTAL 29 vs 29

OK: en todos los meses coinciden el total de la página y lo extraído.


Como verificación adicional, convertimos el texto `Publicado: 27 de febrero de 2022` en una fecha real y revisamos que **cada norma haya sido publicada dentro del mes en que la buscamos**. Si el filtro `desde`/`hasta` no se hubiera aplicado, aparecerían normas de otras fechas.

In [10]:
MESES_ES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
            "agosto": 8, "setiembre": 9, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}


def convertir_fecha(texto):
    # 'Publicado: 27 de febrero de 2022' -> Timestamp('2022-02-27')
    dia, mes, anio = re.search(r"(\d{1,2}) de (\w+) de (\d{4})", texto).groups()
    return pd.Timestamp(int(anio), MESES_ES[mes.lower()], int(dia))


normas["fecha"] = normas["fecha_texto"].apply(convertir_fecha)

# El mes buscado está en los primeros 7 caracteres de la columna "mes" (ej. "2022-02")
normas["fecha_en_su_mes"] = normas["fecha"].dt.strftime("%Y-%m") == normas["mes"].str[:7]
print("Normas publicadas fuera del mes buscado:", (~normas["fecha_en_su_mes"]).sum())
print("Rango de fechas:", normas["fecha"].min().date(), "a", normas["fecha"].max().date())

Normas publicadas fuera del mes buscado: 0
Rango de fechas: 2022-01-06 a 2022-05-26


## Paso 7. Eliminar normas repetidas

Usamos el `enlace` como identificador de cada norma, porque es único para cada página de gob.pe. También revisamos si hay números de norma repetidos.

In [11]:
filas_antes = len(normas)
print("Enlaces repetidos:", normas["enlace"].duplicated().sum())
print("Números de norma repetidos:", normas["numero"].duplicated().sum())

normas = normas.drop_duplicates(subset="enlace").reset_index(drop=True)
print(f"\nFilas antes: {filas_antes} | después: {len(normas)} | repetidas eliminadas: {filas_antes - len(normas)}")

Enlaces repetidos: 0
Números de norma repetidos: 0

Filas antes: 29 | después: 29 | repetidas eliminadas: 0


No hubo normas repetidas. Es lo esperable: los rangos `desde`/`hasta` de los cinco meses no se superponen y cada norma tiene una sola fecha de publicación, así que una norma no puede salir en dos meses. Igual dejamos el paso, porque si se cambian los rangos de búsqueda (por ejemplo, rangos que se crucen) o si el buscador mostrara una norma dos veces, las repetidas inflarían los conteos por departamento.

## Paso 8. Quedarse solo con normas de la PCM

Aunque la búsqueda filtra por PCM, pueden colarse normas de otras instituciones. Nos quedamos solo con las que tienen `/institucion/pcm/` en el enlace.

In [12]:
es_pcm = normas["enlace"].str.contains("/institucion/pcm/", regex=False)

print("Normas que NO son de la PCM (se quitan):", (~es_pcm).sum())
print(normas.loc[~es_pcm, ["numero", "enlace"]].to_string())

normas = normas[es_pcm].reset_index(drop=True)
print(f"\nNormas de la PCM que quedan: {len(normas)}")

Normas que NO son de la PCM (se quitan): 0
Empty DataFrame
Columns: [numero, enlace]
Index: []

Normas de la PCM que quedan: 29


En nuestra temporada (2022) **no se coló ninguna norma de otra institución**: los 29 enlaces tienen `/institucion/pcm/`, así que se quitaron 0. El filtro igual queda en el código porque en otras temporadas sí se cuelan (por ejemplo, resoluciones del OECE).

La *Resolución de Secretaría de Gestión Pública N.° 003-2022-PCM/SGP* sí se queda, porque la Secretaría de Gestión Pública es un órgano de la PCM y su enlace está bajo `/institucion/pcm/`. De todos modos no es un decreto de lluvias, así que se descartará en el paso 11.

## Paso 9. Título completo de cada norma (con `requests` y `BeautifulSoup`)

Los títulos de la lista de resultados vienen cortados con "...". La página de cada norma no necesita Selenium (el título viene en el HTML), así que la leemos con `requests` + `BeautifulSoup`:

- Enviamos `headers={"User-Agent": "Mozilla/5.0"}`, porque gob.pe rechaza los pedidos que no se identifican como navegador.
- El título completo está en `sopa.find("div", class_="description")`. Ese `div` además contiene el bloque de descarga del PDF (`div.institution-document__files`), que es de donde sale el texto extra *"DS N° ... PDF ... Descargar"*. Decidimos **limpiarlo**: borramos ese bloque con `.decompose()` antes de sacar el texto, para que el título quede solo con el título.
- Hacemos una pausa de 1 segundo entre páginas.

In [13]:
def obtener_titulo_completo(enlace):
    respuesta = requests.get(enlace, headers=HEADERS, timeout=30)
    respuesta.raise_for_status()                       # error si la página no respondió 200
    sopa = BeautifulSoup(respuesta.text, "html.parser")
    descripcion = sopa.find("div", class_="description")
    # Quitamos el bloque del PDF ("DS N° ... PDF ... Descargar")
    for bloque in descripcion.find_all("div", class_="institution-document__files"):
        bloque.decompose()
    texto = descripcion.get_text(" ", strip=True)
    texto = re.sub(r"\s+", " ", texto)                 # un solo espacio entre palabras
    return texto.strip(" |")                           # algunos títulos empiezan con "| "


titulos_completos = []
for i, enlace in enumerate(normas["enlace"], start=1):
    try:
        titulo = obtener_titulo_completo(enlace)
    except Exception as error:
        print(f"  Error en {enlace}: {error}")
        titulo = None
    titulos_completos.append(titulo)
    print(f"{i:>2}/{len(normas)}  {normas.loc[i - 1, 'numero']}: {len(titulo or '')} caracteres")
    time.sleep(PAUSA_REQUESTS)   # pausa de 1 segundo entre páginas

normas["titulo"] = titulos_completos

 1/29  Decreto Supremo N.° 011-2022-PCM: 335 caracteres
 2/29  Decreto Supremo N.° 010-2022-PCM: 223 caracteres
 3/29  Decreto Supremo N.° 009-2022-PCM: 222 caracteres
 4/29  Decreto Supremo N.° 004-2022-PCM: 426 caracteres
 5/29  Decreto Supremo N.° 007-2022-PCM: 145 caracteres
 6/29  Decreto Supremo N.° 006-2022-PCM: 359 caracteres
 7/29  Decreto Supremo N.° 005-2022-PCM: 333 caracteres
 8/29  Decreto Supremo N.° 002-2022-PCM: 331 caracteres
 9/29  Decreto Supremo N.° 016-2022-PCM: 234 caracteres
10/29  Decreto Supremo N.° 015-2022-PCM: 335 caracteres
11/29  Decreto Supremo N.° 014-2022-PCM: 214 caracteres
12/29  Decreto Supremo N.° 013-2022-PCM: 449 caracteres
13/29  Decreto Supremo N.° 012-2022-PCM: 281 caracteres
14/29  Decreto Supremo N.° 029-2022-PCM: 214 caracteres
15/29  Decreto Supremo N.° 028-2022-PCM: 192 caracteres
16/29  Decreto Supremo N.° 027-2022-PCM: 144 caracteres
17/29  Decreto Supremo N.° 030-2022-PCM: 394 caracteres
18/29  Resolución de Secretaría de Gestión Públi

**Verificación del título completo:** revisamos que no haya títulos vacíos, que ninguno arrastre el texto "Descargar", y que el título completo sea al menos tan largo como el título cortado de la lista.

In [14]:
print("Títulos vacíos o con error:", normas["titulo"].isna().sum())
print("Títulos que todavía traen 'Descargar':", normas["titulo"].str.contains("Descargar").sum())

normas["largo_corto"] = normas["titulo_corto"].str.len()
normas["largo_completo"] = normas["titulo"].str.len()
print("Títulos completos más cortos que el de la lista:", (normas["largo_completo"] < normas["largo_corto"] - 5).sum())

normas[["numero", "titulo_corto", "titulo", "largo_corto", "largo_completo"]].head(5)

Títulos vacíos o con error: 0
Títulos que todavía traen 'Descargar': 0
Títulos completos más cortos que el de la lista: 0


,numero,titulo_corto,titulo,largo_corto,largo_completo
0,Decreto Supremo N.° 011-2022-PCM,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia ...,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia ...,203,335
1,Decreto Supremo N.° 010-2022-PCM,"Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir ...","Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir ...",202,223
2,Decreto Supremo N.° 009-2022-PCM,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto...","Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto...",203,222
3,Decreto Supremo N.° 004-2022-PCM,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria...","Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria...",203,426
4,Decreto Supremo N.° 007-2022-PCM,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,145,145


## Paso 10. Columnas `es_lluvia`, `tipo` y `motivo`

Usamos el título completo en minúsculas (`.lower()`) y el operador `in`:

- `es_lluvia`: `True` si el título contiene `"lluvia"` o `"precipitaciones"`.
- `tipo`: `"prorroga"` si contiene `"prórroga"` o `"prorroga"`; si no, `"declara"` si contiene `"declara"`; si no, `"otro"`. El orden importa: una prórroga casi siempre también dice "declarado", así que primero se revisa "prórroga".
- `motivo`: `"peligro inminente"` o `"impacto de daños"` si el título contiene esa frase; si no, `"otro"`.

In [15]:
def es_lluvia(titulo):
    t = titulo.lower()
    return ("lluvia" in t) or ("precipitaciones" in t)


def clasificar_tipo(titulo):
    t = titulo.lower()
    if "prórroga" in t or "prorroga" in t:
        return "prorroga"
    elif "declara" in t:
        return "declara"
    else:
        return "otro"


def clasificar_motivo(titulo):
    t = titulo.lower()
    if "peligro inminente" in t:
        return "peligro inminente"
    elif "impacto de daños" in t:
        return "impacto de daños"
    else:
        return "otro"


normas["es_lluvia"] = normas["titulo"].apply(es_lluvia)
normas["tipo"] = normas["titulo"].apply(clasificar_tipo)
normas["motivo"] = normas["titulo"].apply(clasificar_motivo)

# ¿Algún título tiene las dos frases de motivo a la vez? (así sabríamos si el orden del if importa)
ambos = normas["titulo"].str.lower().apply(lambda t: "peligro inminente" in t and "impacto de daños" in t)
print("Títulos con 'peligro inminente' e 'impacto de daños' a la vez:", ambos.sum())
print()
print(normas["es_lluvia"].value_counts(), "\n")
print(normas["tipo"].value_counts(), "\n")
print(normas["motivo"].value_counts())

Títulos con 'peligro inminente' e 'impacto de daños' a la vez: 0

es_lluvia
False    28
True      1
Name: count, dtype: int64 

tipo
prorroga    17
declara     11
otro         1
Name: count, dtype: int64 

motivo
otro                22
impacto de daños     7
Name: count, dtype: int64


### Revisión de las filas clasificadas como `"otro"`

Revisamos una por una las normas que quedaron con `tipo == "otro"` o `motivo == "otro"`, junto con `es_lluvia`, para ver si están bien clasificadas.

In [16]:
otros = normas[(normas["tipo"] == "otro") | (normas["motivo"] == "otro")]
print(f"Filas con tipo 'otro' o motivo 'otro': {len(otros)}\n")
for _, fila in otros.iterrows():
    print(f"{fila['numero']} | es_lluvia={fila['es_lluvia']} | tipo={fila['tipo']} | motivo={fila['motivo']}")
    print(f"   {fila['titulo']}\n")

Filas con tipo 'otro' o motivo 'otro': 22

Decreto Supremo N.° 011-2022-PCM | es_lluvia=False | tipo=declara | motivo=otro
   Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social

Decreto Supremo N.° 010-2022-PCM | es_lluvia=False | tipo=prorroga | motivo=otro
   Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta el 13 de febrero de 2022 .

Decreto Supremo N.° 004-2022-PCM | es_lluvia=False | tipo=prorroga | motivo=otro
   Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honor

Dos revisiones más sobre la clasificación:

1. **Normas con `motivo` distinto de `"otro"` que NO son de lluvias:** ¿por qué hay normas "por impacto de daños" que no son de lluvia?
2. **¿Se nos escapa alguna norma de lluvias?** Buscamos en las normas con `es_lluvia == False` otras palabras que suelen aparecer en emergencias por lluvias (*pluvial, huaico, inundación, deslizamiento, desborde, Niño*).

In [17]:
print("1) Normas con motivo 'peligro inminente' o 'impacto de daños' que no son de lluvias:\n")
motivo_no_lluvia = normas[(normas["motivo"] != "otro") & (~normas["es_lluvia"])]
for _, fila in motivo_no_lluvia.iterrows():
    print(f"{fila['numero']} | tipo={fila['tipo']} | motivo={fila['motivo']}")
    print(f"   {fila['titulo']}\n")

palabras_lluvia = ["pluvial", "huaico", "huayco", "inundaci", "desliza", "desborde", "niño"]
posibles = normas[(~normas["es_lluvia"]) &
                  normas["titulo"].str.lower().apply(lambda t: any(p in t for p in palabras_lluvia))]
print("2) Normas con es_lluvia == False que mencionan otras palabras de lluvias:", len(posibles))

1) Normas con motivo 'peligro inminente' o 'impacto de daños' que no son de lluvias:

Decreto Supremo N.° 009-2022-PCM | tipo=prorroga | motivo=impacto de daños
   Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto y San Martín, por impacto de daños ante la ocurrencia de movimiento sísmico

Decreto Supremo N.° 014-2022-PCM | tipo=prorroga | motivo=impacto de daños
   Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca y San Martín, por impacto de daños ante la ocurrencia de movimiento sísmico

Decreto Supremo N.° 029-2022-PCM | tipo=prorroga | motivo=impacto de daños
   Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca y San Martín, por impacto de daños ante la ocurrencia de movimiento sísmico

Decreto Supremo N.° 028-

**¿Están bien clasificadas las filas que quedaron como `"otro"`?**

- **`tipo == "otro"` (1 fila):** la *Resolución de Secretaría de Gestión Pública N.° 003-2022-PCM/SGP* no declara ni prorroga un estado de emergencia. Deja sin efecto una resolución anterior y ajusta el horario de los Centros MAC "durante el estado de emergencia sanitaria". **Está bien clasificada** como "otro".
- **`motivo == "otro"` (22 filas):** ninguno de estos títulos contiene "peligro inminente" ni "impacto de daños". Son emergencias por la COVID-19 (el Estado de Emergencia Nacional y sus prórrogas o modificaciones), por orden interno (Lima Metropolitana y Callao, provincias de La Libertad), la Red Vial Nacional y prórrogas en Loreto, Madre de Dios, Huánuco/Pasco/Ucayali, Apurímac y zonas de Ayacucho, Huancavelica, Cusco y Junín cuyo título **no dice el motivo**. Como el título no tiene ninguna de las dos frases, **"otro" es correcto**. Además, ninguna de estas 22 normas es de lluvias, así que no afectan el resultado final.

**Lo que la regla de `tipo` no resuelve bien:** cuatro decretos que **modifican** el decreto de la COVID-19 (002, 005, 011 y 015-2022-PCM) quedaron como `"declara"`. No declaran una emergencia nueva: la palabra "declara" aparece porque citan el nombre completo del *"Decreto Supremo N° 184-2020-PCM, Decreto Supremo que declara Estado de Emergencia Nacional…"*. Es una limitación de clasificar solo buscando palabras. No cambia nuestros resultados porque ninguno es de lluvias y se descartan en el paso 11. En cambio, *"Declarar por término de cuarenta y cinco (45) días…"* (012-2022-PCM) sí es una declaratoria y queda bien como "declara", porque "declara" está dentro de "declarar".

**Normas "por impacto de daños" que no son de lluvias:** 6 de las 7 normas con `motivo == "impacto de daños"` son **por movimiento sísmico** (el sismo que afectó Amazonas, Cajamarca, Loreto y San Martín, y el de Caylloma en Arequipa). Su `motivo` está bien, y `es_lluvia == False` también. Esto muestra por qué hay que filtrar por `es_lluvia`: el motivo no dice cuál fue el peligro.

**¿Se nos escapó alguna norma de lluvias?** No: ninguna norma con `es_lluvia == False` menciona otras palabras típicas de lluvias (pluvial, huaico, inundación, deslizamiento, desborde, Niño).

## Paso 11. Quedarse solo con las normas de lluvias

In [18]:
lluvias = normas[normas["es_lluvia"]].reset_index(drop=True)
print("Normas de lluvias:", len(lluvias))
print(lluvias["tipo"].value_counts().to_string(), "\n")
lluvias[["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]].head(5)

Normas de lluvias: 1
tipo
declara    1 



,numero,fecha,titulo,tipo,motivo,enlace
0,Decreto Supremo N.° 032-2022-PCM,2022-04-01,"Decreto Supremo que declara el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Ayacucho y Piura, ...",declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-legales/2929443-032-2022-pcm


### Verificación: ¿de verdad hay una sola norma de lluvias en la temporada 2022?

Solo **1** de las 29 normas es de lluvias, así que `head(5)` muestra una sola fila porque no hay más. Como es un resultado sorprendente, comprobamos que no sea un error de nuestro código. Volvemos a buscar en gob.pe (misma institución, mismas fechas: enero a mayo de 2022) con términos más específicos: `precipitaciones` y `lluvias`. Son dos páginas más, con la misma pausa de 3 segundos.

In [19]:
driver = webdriver.Edge(options=opciones)
espera = WebDriverWait(driver, 30)

for termino in ["precipitaciones", "lluvias"]:
    url_termino = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
                   f"&term={termino}&sort_by=recent")
    abrir_busqueda("01-01-2022", "31-05-2022", url_base=url_termino)
    encontrados = [fila["numero"] for fila in extraer_resultados()]
    print(f"term={termino!r}: {leer_total()} resultado(s) -> {encontrados}")
    time.sleep(PAUSA_SELENIUM)

driver.quit()

term='precipitaciones': 1 resultado(s) -> ['Decreto Supremo N.° 032-2022-PCM']
term='lluvias': 0 resultado(s) -> []


El buscador de gob.pe confirma el resultado: entre enero y mayo de 2022, la única norma de la PCM que menciona "precipitaciones" es el **Decreto Supremo N.° 032-2022-PCM**, y ninguna menciona "lluvias". Así que no es un error de nuestro código: es lo que **gob.pe tiene publicado** para esta temporada. Es una **limitación importante** para la Parte 3: en 2022 casi todas las normas de emergencia que encontramos son por COVID-19, por seguridad o por sismos, y el cruce con la lluvia se basará en un solo decreto (Amazonas, Ayacucho y Piura).

## Paso 12. Departamentos que menciona cada decreto

Buscamos cada departamento en el **título original (con mayúsculas)** y **por palabra completa**, con `re.search(r"\b" + departamento + r"\b", titulo)`. `\b` es un "borde de palabra": exige que antes y después del nombre no haya otra letra, así `Ica` no se encuentra dentro de `Huancavelica`.

La función devuelve una **lista sin repetidos**: recorre los 25 departamentos una sola vez y agrega cada uno como máximo una vez, aunque el nombre aparezca varias veces en el título.

In [20]:
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                 "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]


def departamentos_mencionados(titulo):
    encontrados = []
    for departamento in departamentos:
        if re.search(r"\b" + departamento + r"\b", titulo):
            encontrados.append(departamento)   # cada departamento se agrega una sola vez
    return encontrados


lluvias["departamentos"] = lluvias["titulo"].apply(departamentos_mencionados)
lluvias["n_departamentos"] = lluvias["departamentos"].apply(len)

print("Decretos de lluvias sin ningún departamento encontrado:", (lluvias["n_departamentos"] == 0).sum())
lluvias[["numero", "tipo", "departamentos"]]

Decretos de lluvias sin ningún departamento encontrado: 0


,numero,tipo,departamentos
0,Decreto Supremo N.° 032-2022-PCM,declara,"[Amazonas, Ayacucho, Piura]"


### Verificación: ¿dónde se habría contado mal?

Nuestro único decreto de lluvias (032-2022-PCM) no tiene ninguna trampa: menciona Amazonas, Ayacucho y Piura, cada uno una vez. Para comprobar que el método funciona con casos difíciles, aplicamos la misma función a **las 29 normas de la PCM de nuestra temporada** (no solo a las de lluvias) y la comparamos con dos formas de contar mal:

- **Método ingenuo:** `departamento.lower() in titulo.lower()` (buscar en minúsculas, sin palabra completa).
- **Contar apariciones:** sumar cada vez que el nombre aparece en el título, en lugar de contar el departamento una sola vez por decreto.

In [21]:
def departamentos_ingenuo(titulo):
    return [d for d in departamentos if d.lower() in titulo.lower()]


problemas = []
for _, fila in normas.iterrows():
    correctos = departamentos_mencionados(fila["titulo"])
    for d in departamentos_ingenuo(fila["titulo"]):
        if d not in correctos:
            problemas.append({"numero": fila["numero"], "departamento": d,
                              "problema": "el método ingenuo lo encuentra, pero el decreto no lo menciona"})
    for d in correctos:
        n_apariciones = len(re.findall(r"\b" + d + r"\b", fila["titulo"]))
        if n_apariciones > 1:
            problemas.append({"numero": fila["numero"], "departamento": d,
                              "problema": f"aparece {n_apariciones} veces en el título (lo contamos 1 sola vez)"})

pd.DataFrame(problemas)

,numero,departamento,problema
0,Decreto Supremo N.° 011-2022-PCM,Ica,"el método ingenuo lo encuentra, pero el decreto no lo menciona"
1,Decreto Supremo N.° 010-2022-PCM,Ica,"el método ingenuo lo encuentra, pero el decreto no lo menciona"
2,Decreto Supremo N.° 006-2022-PCM,Madre de Dios,aparece 2 veces en el título (lo contamos 1 sola vez)
3,Decreto Supremo N.° 005-2022-PCM,Ica,"el método ingenuo lo encuentra, pero el decreto no lo menciona"
4,Decreto Supremo N.° 002-2022-PCM,Ica,"el método ingenuo lo encuentra, pero el decreto no lo menciona"
5,Decreto Supremo N.° 015-2022-PCM,Ica,"el método ingenuo lo encuentra, pero el decreto no lo menciona"
6,Decreto Supremo N.° 013-2022-PCM,Ica,"el método ingenuo lo encuentra, pero el decreto no lo menciona"
7,Decreto Supremo N.° 013-2022-PCM,Cusco,aparece 2 veces en el título (lo contamos 1 sola vez)
8,Decreto Supremo N.° 012-2022-PCM,Lima,aparece 2 veces en el título (lo contamos 1 sola vez)
9,Decreto Supremo N.° 030-2022-PCM,Ica,"el método ingenuo lo encuentra, pero el decreto no lo menciona"


In [22]:
# El caso más claro de nuestra temporada: el Decreto Supremo N.° 013-2022-PCM
ejemplo = normas[normas["numero"].str.contains("013-2022-PCM")].iloc[0]
print(ejemplo["numero"], "-", ejemplo["fecha"].date())
print(ejemplo["titulo"], "\n")
print("Método ingenuo (minúsculas + in):", departamentos_ingenuo(ejemplo["titulo"]))
print("Nuestro método (\\b + sin repetir):", departamentos_mencionados(ejemplo["titulo"]))
print("¿'ica' está dentro de 'huancavelica'?:", "ica" in "huancavelica")
print()
# Y el caso del enunciado: "Ica" como provincia y como departamento en el mismo título
frase = "provincia de Ica del departamento de Ica"
print(f"'{frase}' ->", departamentos_mencionados(frase))

Decreto Supremo N.° 013-2022-PCM - 2022-02-10
Prórroga del Estado de Emergencia en distritos de las provincias de Huanta y La Mar del departamento de Ayacucho; de las provincias de Tayacaja y Churcampa del departamento de Huancavelica; de la provincia de La Convención del departamento de Cusco; de las provincias de Satipo, Concepción y Huancayo del departamento de Junín; y, en el Centro Poblado de Yuveni en el distrito de Vilcabamba de la provincia de La Convención del departamento de Cusco 

Método ingenuo (minúsculas + in): ['Ayacucho', 'Cusco', 'Huancavelica', 'Ica', 'Junín']
Nuestro método (\b + sin repetir): ['Ayacucho', 'Cusco', 'Huancavelica', 'Junín']
¿'ica' está dentro de 'huancavelica'?: True

'provincia de Ica del departamento de Ica' -> ['Ica']


**¿Cómo evitamos contar mal? Ejemplo de nuestra temporada: Decreto Supremo N.° 013-2022-PCM (publicado el 10 de febrero de 2022).**

Su título menciona cuatro departamentos: **Ayacucho, Huancavelica, Cusco y Junín**.

1. **Ica dentro de Huancavelica.** Con el método ingenuo (título en minúsculas + `in`), el decreto también "menciona" **Ica**, porque `"ica"` está dentro de `"huancavel`**`ica`**`"`. Ica habría sumado una emergencia que nunca tuvo. En nuestra temporada el problema es aún más grande: `"ica"` también está dentro de palabras comunes como "modif**ica**" (002, 005, 010, 011, 015 y 030-2022-PCM) y "Públ**ica**" (003-2022-PCM/SGP). Para evitarlo buscamos en el **título original** con `re.search(r"\b" + departamento + r"\b", titulo)`. Así "Ica" solo cuenta si aparece como **palabra completa y con mayúscula**, y todos esos falsos positivos desaparecen (el método correcto devuelve `['Ayacucho', 'Cusco', 'Huancavelica', 'Junín']`).
2. **Un departamento que aparece dos veces.** En el mismo decreto, "departamento de **Cusco**" aparece **dos veces**. Si contáramos apariciones, Cusco sumaría 2 por un solo decreto. Nuestra función recorre la lista de los 25 departamentos **una sola vez** y agrega cada uno como máximo una vez, así que Cusco cuenta **1**. Es el mismo caso que *"provincia de Ica del departamento de Ica"*: la prueba de la última línea devuelve `['Ica']` una sola vez. En nuestra temporada, el caso más parecido es el de 006 y 055-2022-PCM: "Madre de Dios" aparece como **distrito** (*"distritos de Fitzcarrald, Manu, Madre de Dios y Huepetuhe"*) y como **departamento** (*"del departamento de Madre de Dios"*), y lo contamos una sola vez.

Nuestro único decreto de lluvias (032-2022-PCM) no tiene ninguna de estas trampas, pero la verificación muestra que el método las resuelve.

**Lo que este método no resuelve:** solo encuentra los departamentos que el título nombra. Además, fallaría si un nombre de departamento apareciera como provincia o distrito **de otro departamento**. Por ejemplo, la provincia de Ucayali pertenece al departamento de **Loreto**: *"provincia de Ucayali del departamento de Loreto"* sumaría también a Ucayali. En nuestros decretos de lluvias no pasa, pero hay que revisarlo en otras temporadas.

### Tabla por departamento

Para cada uno de los 25 departamentos contamos cuántos decretos de lluvias lo mencionan, separando **declaratorias** (`tipo == "declara"`) y **prórrogas** (`tipo == "prorroga"`). Incluimos los 25 departamentos, aunque tengan 0, para que ninguno desaparezca en el cruce de la Parte 3.

In [23]:
filas_departamento = []
for departamento in departamentos:
    lo_menciona = lluvias["departamentos"].apply(lambda lista: departamento in lista)
    filas_departamento.append({
        "departamento": departamento,
        "declaratorias": (lo_menciona & (lluvias["tipo"] == "declara")).sum(),
        "prorrogas": (lo_menciona & (lluvias["tipo"] == "prorroga")).sum(),
    })

por_departamento = pd.DataFrame(filas_departamento)
print(por_departamento.sort_values(["declaratorias", "prorrogas"], ascending=False).to_string(index=False))

 departamento  declaratorias  prorrogas
     Amazonas              1          0
     Ayacucho              1          0
        Piura              1          0
       Áncash              0          0
     Apurímac              0          0
     Arequipa              0          0
    Cajamarca              0          0
       Callao              0          0
        Cusco              0          0
 Huancavelica              0          0
      Huánuco              0          0
          Ica              0          0
        Junín              0          0
  La Libertad              0          0
   Lambayeque              0          0
         Lima              0          0
       Loreto              0          0
Madre de Dios              0          0
     Moquegua              0          0
        Pasco              0          0
         Puno              0          0
   San Martín              0          0
        Tacna              0          0
       Tumbes              0          0


**Verificación de la tabla:** la suma de declaratorias por departamento debe ser igual a la suma, decreto por decreto, del número de departamentos que menciona cada declaratoria (lo mismo para las prórrogas).

In [24]:
for tipo, columna in [("declara", "declaratorias"), ("prorroga", "prorrogas")]:
    esperado = lluvias.loc[lluvias["tipo"] == tipo, "n_departamentos"].sum()
    obtenido = por_departamento[columna].sum()
    print(f"{columna}: suma en la tabla = {obtenido} | suma decreto por decreto = {esperado} | coinciden: {obtenido == esperado}")
    assert obtenido == esperado

print("\nDepartamentos con al menos una declaratoria o prórroga:",
      ((por_departamento["declaratorias"] + por_departamento["prorrogas"]) > 0).sum(), "de", len(por_departamento))

declaratorias: suma en la tabla = 3 | suma decreto por decreto = 3 | coinciden: True
prorrogas: suma en la tabla = 0 | suma decreto por decreto = 0 | coinciden: True

Departamentos con al menos una declaratoria o prórroga: 3 de 25


## Paso 13. Guardar los archivos

- `datos/decretos_lluvias.csv`: una fila por decreto de lluvias, con `numero`, `fecha`, `titulo`, `tipo`, `motivo` y `enlace`. La `fecha` se guarda en formato `AAAA-MM-DD` (la convertimos en el paso 6) y el `titulo` es el título completo ya limpio.
- `datos/decretos_por_departamento.csv`: una fila por departamento (25), con `departamento`, `declaratorias` y `prorrogas`.

Guardamos con `encoding="utf-8-sig"` para que las tildes se vean bien en Excel.

In [25]:
os.makedirs("datos", exist_ok=True)

decretos_lluvias = lluvias[["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]].copy()
decretos_lluvias["fecha"] = decretos_lluvias["fecha"].dt.strftime("%Y-%m-%d")

decretos_lluvias.to_csv("datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")
por_departamento.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")

# Verificación: volvemos a leer los archivos guardados
for archivo in ["datos/decretos_lluvias.csv", "datos/decretos_por_departamento.csv"]:
    df = pd.read_csv(archivo, encoding="utf-8-sig")
    print(f"{archivo}: {df.shape[0]} filas x {df.shape[1]} columnas -> {list(df.columns)}")

datos/decretos_lluvias.csv: 1 filas x 6 columnas -> ['numero', 'fecha', 'titulo', 'tipo', 'motivo', 'enlace']
datos/decretos_por_departamento.csv: 25 filas x 3 columnas -> ['departamento', 'declaratorias', 'prorrogas']


In [26]:
pd.read_csv("datos/decretos_lluvias.csv", encoding="utf-8-sig").head()

,numero,fecha,titulo,tipo,motivo,enlace
0,Decreto Supremo N.° 032-2022-PCM,2022-04-01,"Decreto Supremo que declara el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Ayacucho y Piura, ...",declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-legales/2929443-032-2022-pcm


In [27]:
pd.read_csv("datos/decretos_por_departamento.csv", encoding="utf-8-sig")

,departamento,declaratorias,prorrogas
0,Amazonas,1,0
1,Áncash,0,0
2,Apurímac,0,0
3,Arequipa,0,0
4,Ayacucho,1,0
5,Cajamarca,0,0
6,Callao,0,0
7,Cusco,0,0
8,Huancavelica,0,0
9,Huánuco,0,0


## Resumen de la Parte 1

| Paso | Resultado |
|---|---|
| Normas extraídas con Selenium (ene–may 2022) | 29 (8 + 5 + 5 + 4 + 7), coinciden con el total de la página en los 5 meses |
| Normas repetidas eliminadas | 0 |
| Normas de otras instituciones eliminadas | 0 |
| Títulos completos obtenidos con `requests` | 29 de 29 |
| Normas de lluvias | **1**: Decreto Supremo N.° 032-2022-PCM (1 de abril de 2022), declaratoria por impacto de daños a consecuencia de intensas precipitaciones pluviales |
| Departamentos con declaratoria por lluvias | Amazonas, Ayacucho y Piura (1 cada uno); los otros 22 quedan en 0 |

En la temporada 2022, gob.pe tiene publicada **una sola** norma de emergencia por lluvias de la PCM (lo comprobamos también buscando `precipitaciones` y `lluvias`). Este es el insumo para el cruce de la Parte 3, y es una limitación que habrá que tener en cuenta al responder la pregunta del trabajo.